In [1]:
import requests
from bs4 import BeautifulSoup

In [2]:

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/",

}

response = requests.get("https://news.err.ee/", headers=headers)
doc = BeautifulSoup(response.text)

In [3]:
print(response.status_code)

200


In [4]:
# What is the selector for the headlines WITH THE LINKS?
# Works from right to left
# a meana <a> tag
# .c_t means something with the class "c_t"
# (space) means "inside of"
# so this means links inside of the class c_t
links = doc.select(".news-article")
len(links)

20

In [5]:
print (links[2])

<h2 class="news-article"><a aria-label="Tallinn deputy mayor suggests Old Town property sell-offs amid vacancies" href="https://news.err.ee/1610080954/tallinn-deputy-mayor-suggests-old-town-property-sell-offs-amid-vacancies">Tallinn deputy mayor suggests Old Town property sell-offs amid vacancies</a></h2>


In [6]:
# We love a good CSV file
# we probably want one with a HEADLINE column
# and also a URL column

# to make a CSV we need a LIST of DICTIONARIES
for link in links:
    print("----")
    # Print out the text
    #print(link.text)
    # Print out the URL it points to
    # <a href="blahblah.html" title="...">
    # <img src="....">    
    #print(link['href'])
    a = link.find("a")
    print(a.text)
    print(a['href'])

----
Toyota driver Sami Pajari wins Rally Estonia after dominating weekend
https://news.err.ee/1610080450/toyota-driver-sami-pajari-wins-rally-estonia-after-dominating-weekend
----
Gallery: XXI Hiiu Folk festival day one
https://news.err.ee/1610081005/gallery-xxi-hiiu-folk-festival-day-one
----
Tallinn deputy mayor suggests Old Town property sell-offs amid vacancies
https://news.err.ee/1610080954/tallinn-deputy-mayor-suggests-old-town-property-sell-offs-amid-vacancies
----
35th anniversary of restoration of Estonian independence to see extensive events
https://news.err.ee/1610080936/35th-anniversary-of-restoration-of-estonian-independence-to-see-extensive-events
----
Four treated for carbon monoxide poisoning after visiting Tallinn spa
https://news.err.ee/1610080909/four-treated-for-carbon-monoxide-poisoning-after-visiting-tallinn-spa
----
Estonia getting new British deployment tailored to its battlefield needs
https://news.err.ee/1610080894/estonia-getting-new-british-deployment-tailo

In [7]:
# We love a good CSV file
# we probably want one with a HEADLINE column
# and also a URL column

# to make a CSV we need a LIST of DICTIONARIES
# YOUR MISSION:
# 1. Make an empty list called all_data
# 2. Add our data dictionary to it each time we loop
# 3. Look at the list
all_data = []
for link in links:
    # print("----")
    a = link.find("a")
    data = {
        'url': a['href'],
        'headline': a.text
    }
    # print(data)
    all_data.append(data)
all_data

[{'url': 'https://news.err.ee/1610080450/toyota-driver-sami-pajari-wins-rally-estonia-after-dominating-weekend',
  'headline': 'Toyota driver Sami Pajari wins Rally Estonia after dominating weekend'},
 {'url': 'https://news.err.ee/1610081005/gallery-xxi-hiiu-folk-festival-day-one',
  'headline': 'Gallery: XXI Hiiu Folk festival day one'},
 {'url': 'https://news.err.ee/1610080954/tallinn-deputy-mayor-suggests-old-town-property-sell-offs-amid-vacancies',
  'headline': 'Tallinn deputy mayor suggests Old Town property sell-offs amid vacancies'},
 {'url': 'https://news.err.ee/1610080936/35th-anniversary-of-restoration-of-estonian-independence-to-see-extensive-events',
  'headline': '35th anniversary of restoration of Estonian independence to see extensive events'},
 {'url': 'https://news.err.ee/1610080909/four-treated-for-carbon-monoxide-poisoning-after-visiting-tallinn-spa',
  'headline': 'Four treated for carbon monoxide poisoning after visiting Tallinn spa'},
 {'url': 'https://news.err.e

In [8]:
import pandas as pd

# Just feed the list of dictionaries to pandas
# and it makes us a nice beautiful dataframe
df = pd.DataFrame(all_data)
df.head()

,url,headline
0,https://news.err.ee/1610080450/toyota-driver-s...,Toyota driver Sami Pajari wins Rally Estonia a...
1,https://news.err.ee/1610081005/gallery-xxi-hii...,Gallery: XXI Hiiu Folk festival day one
2,https://news.err.ee/1610080954/tallinn-deputy-...,Tallinn deputy mayor suggests Old Town propert...
3,https://news.err.ee/1610080936/35th-anniversar...,35th anniversary of restoration of Estonian in...
4,https://news.err.ee/1610080909/four-treated-fo...,Four treated for carbon monoxide poisoning aft...


In [9]:
import os

# Try to create a folder called 'data'
# and if it exists DON'T THROW AN ERROR
os.makedirs("data\err_ee", exist_ok=True)

<>:5: SyntaxWarning: "\e" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\e"? A raw string is also an option.
<>:5: SyntaxWarning: "\e" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\e"? A raw string is also an option.
C:\Users\jpjul\AppData\Local\Temp\ipykernel_27988\771705377.py:5: SyntaxWarning: "\e" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\e"? A raw string is also an option.
  os.makedirs("data\err_ee", exist_ok=True)


In [10]:
from datetime import datetime

# This would keep track down to the second
datetime.now().strftime("%Y-%m-%d_%H.%M.%S")

# This only does the day
date_string = datetime.now().strftime("%Y-%m-%d_%H.%M.%S")
filepath = f"data/err_ee/{date_string}.csv"

df.to_csv(filepath, index=False)

In [ ]:
# Append to an always-updated file, deduplicated by url.

# add columns for when we scraped this
df['scrape_date'] = datetime.now().strftime("%Y-%m-%d")
df['scrape_datetime'] = datetime.now().strftime("%Y-%m-%d_%H.%M.%S")

# open the always-updated file if it exists, else start blank
try:
    existing_df = pd.read_csv("data/err_ee-always-updated.csv")
except:
    existing_df = pd.DataFrame([])

# OLD first, then NEW, so keep='first' preserves the earliest scrape date
combined = pd.concat([existing_df, df], ignore_index=True)
print("Before dropping duplicates:", len(combined))

# dedup on url ONLY -> only genuinely new articles get added to the file
combined = combined.drop_duplicates(subset=['url'], keep='first')
print("After dropping duplicates: ", len(combined))

combined.to_csv("data/err_ee-always-updated.csv", index=False)
combined.head()